<a href="https://colab.research.google.com/github/luzcielponce-create/MS-E-226-Project-/blob/main/Final_Copy_of_TV_Shows_%26_Movies_Streaming_Data_Merged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setting up!

In [2]:
import os
import csv
import re
import unicodedata #handle special characters
from collections import defaultdict
from pathlib import Path

import pandas as pd
from google.colab import files
from IPython.display import display

# ------------------------------------------------------------
# Make sure all data files are in Colab
# Send error if any is missing
# ------------------------------------------------------------
os.chdir("/content") #work inside Colab's main storage folder
folder = Path("/content") #save folder as "Path" object to build file paths later

required_files = [
    "MoviesOnStreamingPlatforms.csv",   # movie catalog
    "tv_shows.csv",                     # TV show catalog
    "netflix_titles.csv",           # Netflix catalog (has "date added")
    "title.basics.tsv.gz",              # IMDb titles (used to find IMDb IDs)
    "title.ratings.tsv.gz",             # IMDb ratings and vote counts
]

missing = []
for name in required_files:
    file_path = folder / name #build full path (ex. /content/tv_shows.csv)
    if not file_path.is_file():
        missing.append(name)   # add name to "missing" list

# If list is still empty, everything is here and we skip this block
# If it has anything in it, stop and say exactly what's wrong
if missing:
    raise FileNotFoundError(
        "Still missing: " + ", ".join(missing)
    )

# Only reach this line if nothing was missing
print("All source files are ready.")

All source files are ready.


**Part 1: Read movie and TV files and combine them**

In [3]:
movies = pd.read_csv(
    folder / "MoviesOnStreamingPlatforms.csv",
    dtype=str,
    keep_default_na=False,
)

tv = pd.read_csv(
    folder / "tv_shows.csv",
    dtype=str,
    keep_default_na=False,
)
tv = tv.drop(columns=["IMDb"])

netflix = pd.read_csv(
    folder / "netflix_titles.csv",
    dtype=str,
    keep_default_na=False,
)

# Stack movies and TV shows into one table
main = pd.concat(
    [movies, tv],
    ignore_index=True,
    sort=False,
).fillna("")

# Add a "content_type" column: is each row a Movie or a TV Show?
content_labels = []

# Add one "Movie" label for every row in the movies table.
for _ in range(len(movies)):
    content_labels.append("Movie")

# Add one "TV Show" label for every row in the TV table.
for _ in range(len(tv)):
    content_labels.append("TV Show")

# Attach the labels as a new column
main["content_type"] = content_labels

# Add a "record_id" column: a unique ID for every row
record_ids = []
for i in range(len(movies)):
    record_ids.append(f"movie_{i + 1}")   # f"..." inserts the value of i + 1
for i in range(len(tv)):
    record_ids.append(f"tv_{i + 1}")

# Attach IDs as a new column
main["record_id"] = record_ids

# working copy
result = main.copy()

# Summary
print(f"Movies:   {len(movies):,}")
print(f"TV shows: {len(tv):,}")
print(f"Total:    {len(main):,}")

Movies:   9,515
TV shows: 5,368
Total:    14,883


**Part 3: Create matching keys: content type + title + year**

In [4]:
# Clean up a title so spelling differences don't matter
def normalize_title(value):
    # Safety checks
    if pd.isna(value):
        return ""

    text = str(value)
    text = unicodedata.normalize("NFKC", text) #standardize special characters
    text = text.casefold() #lowercase everything (casefold = stronger "lower()")

    cleaned_characters = []
    for character in text:
        if character.isalnum(): #true for letters and digits
            cleaned_characters.append(character)
        else:
            cleaned_characters.append(" ")
    text = "".join(cleaned_characters)
    text = re.sub(r"\s+", " ", text) #make any space single
    return text.strip()


# Build one key per row
# Returns list of keys, one per row (ex. "Movie", "inception", 2010)
def make_keys(types, titles, years):
    numeric_years = pd.to_numeric(years, errors="coerce") #convert years from text to numbers
    keys = []

    # zip() walks through the three lists side by side, so on each pass
    # we get the matching type, title, and year for one row
    for kind, title, year in zip(types, titles, numeric_years):
        clean_title = normalize_title(title)

        # A row is unusable if the title is empty, the year is missing,
        # or the year isn't a whole number (e.g. 2010.5).
        if clean_title == "" or pd.isna(year) or not float(year).is_integer():
            keys.append(None)      # None = "this row can't be matched"
        else:
            keys.append((kind, clean_title, int(year)))

    return keys


# Build the key for every row in combined table
main_keys = make_keys(
    main["content_type"],   # "Movie" or "TV Show"
    main["Title"],          # title column
    main["Year"],           # year column
)

target_keys = set(main_keys) - {None}

# Collect just the years that appear in data, as text ("2010")
target_years = set()
for key in target_keys:
    target_years.add(str(key[2]))   # key[2] is the year


# Translate IMDb's categories into movie or tv show
imdb_types = {
    "movie": "Movie",
    "tvMovie": "Movie",
    "video": "Movie",
    "short": "Movie",
    "tvShort": "Movie",
    "tvSeries": "TV Show",
    "tvMiniSeries": "TV Show",
}

**Part 4: Match titles to IMDb IDs**

In [5]:
imdb_candidates = defaultdict(set)
imdb_info = {}   #runtime, genres

# How many IMDb rows we've looked at so far
scanned = 0

print("Scanning IMDb movies and series...")
print("This may take several minutes.")

# Loop through chunks of 200,000 rows
for chunk in pd.read_csv(
    folder / "title.basics.tsv.gz",
    sep="\t",
    compression="gzip",               # read the .gz file without unzipping it
    usecols=[
        "tconst",                     #IMDb ID (tt1375666)
        "titleType",                  #movie, tvSeries, tvEpisode, etc
        "primaryTitle",               #main title
        "originalTitle",              #title in original language
        "startYear",                  #release year
        "runtimeMinutes",
        "genres",
    ],
    dtype=str,
    keep_default_na=False,
    na_values=[r"\N"],
    quoting=csv.QUOTE_NONE,           #ignore quote marks
    chunksize=200_000,
):

    # Add this chunk's row count to total
    scanned += len(chunk)

    # Keep only IMDb rows that could possibly match one of our titles
    is_wanted_type = chunk["titleType"].isin(list(imdb_types))
    is_wanted_year = chunk["startYear"].isin(target_years)
    relevant = chunk.loc[is_wanted_type & is_wanted_year]

    # Translate IMDb's categories into movie or tv show
    content_types = relevant["titleType"].map(imdb_types)

    # IMDb gives two titles per row, try each one as a possible match
    for title_column in ["primaryTitle", "originalTitle"]:
        # Build a key for every relevant IMDb row
        keys = make_keys(
            content_types,
            relevant[title_column],
            relevant["startYear"],
        )
        # Walk through each IMDb key together with its IMDb ID
        for key, imdb_id, runtime, genre in zip(keys, relevant["tconst"], relevant["runtimeMinutes"], relevant["genres"]):
            # If this key also exists in OUR data, remember this ID as a candidate
            if key in target_keys:
                imdb_candidates[key].add(imdb_id)
                imdb_info[imdb_id] = (
                    "" if pd.isna(runtime) else runtime,
                    "" if pd.isna(genre) else genre,
                )

    # Print a progress message every 2 million rows
    if scanned % 2_000_000 == 0:
        print(f"Scanned {scanned:,} IMDb rows...", flush=True)

print(f"IMDb scan finished: {scanned:,} rows.")


#Keep only matches that are unambiguous
unique_ids = {}

# Go through every key and its set of candidate IDs
for key, ids in imdb_candidates.items():

    # len(ids) == 1 means exactly one IMDb title fit this key.
    # If two or more fit (ex. two different films named "Crash" from the same year), don't guess and skip
    if len(ids) == 1:
        # next(iter(ids)) pulls the single item out of a one-item set.
        unique_ids[key] = next(iter(ids))


# Give every row in table its IMDb ID
matched_ids = []

for key in main_keys:
    matched_ids.append(unique_ids.get(key))


# Set of all IMDb IDs we matched
needed_ids = set(unique_ids.values())

Scanning IMDb movies and series...
This may take several minutes.
Scanned 2,000,000 IMDb rows...
Scanned 4,000,000 IMDb rows...
Scanned 6,000,000 IMDb rows...
Scanned 8,000,000 IMDb rows...
Scanned 10,000,000 IMDb rows...
Scanned 12,000,000 IMDb rows...
IMDb scan finished: 12,913,556 rows.


In [6]:
print(len(imdb_candidates), "keys had at least one IMDb candidate")
print(len(unique_ids), "keys had exactly one")
print(sum(1 for i in matched_ids if i is not None), "of", len(matched_ids), "rows got an IMDb ID")

11952 keys had at least one IMDb candidate
11160 keys had exactly one
11160 of 14883 rows got an IMDb ID


**Part 5: Retrieve IMDb ratings & vote counts**

In [7]:
print("Reading IMDb ratings...")

# Dictionary that remembers each matched title's rating info (ex. "tt1375666" -> (8.8, 2500000))
ratings = {}

# Read in chunks of 200,000 rows
for chunk in pd.read_csv(
    folder / "title.ratings.tsv.gz",
    sep="\t",
    compression="gzip",
    dtype={
        "tconst": str,
        "averageRating": float,        #rating becomes a decimal number
        "numVotes": int,               #vote count becomes a whole number
    },
    chunksize=200_000,
):

    # Keep only the rows whose IMDb ID is one we matched
    selected = chunk.loc[chunk["tconst"].isin(needed_ids)]

    for row in selected.itertuples(index=False):
        # Save rating and vote count, filed under the IMDb ID.
        ratings[row.tconst] = (row.averageRating, row.numVotes)


# Build three new columns, one value per row of "result"
rating_values = []
vote_values = []

for imdb_id in matched_ids:
    if imdb_id is None or imdb_id not in ratings:
        rating_values.append(None)
        vote_values.append(None)
    # Unpack the saved (rating, votes) pair and add
    else:
        rating, votes = ratings[imdb_id]
        rating_values.append(rating)
        vote_values.append(votes)

# Add columns to the table
result["imdb_id"] = pd.array(matched_ids, dtype="string")
result["imdb_rating"] = pd.array(rating_values, dtype="Float64")
result["imdb_num_votes"] = pd.array(vote_values, dtype="Int64")


# Add a status column that explains what happened to each row
# A function that looks at one key and returns a status word
def imdb_status(key):
    # Look up every IMDb ID that fit this key
    candidates = imdb_candidates.get(key, set())

    # Nothing in IMDb fit this title/year
    if len(candidates) == 0:
        return "no_match"

    # More than one IMDb title fit, so we refused to guess
    if len(candidates) > 1:
        return "ambiguous"

    # Exactly one candidate, pull it out of the set
    imdb_id = next(iter(candidates))

    # Found the title, but IMDb has no rating for it
    if imdb_id not in ratings:
        return "matched_no_rating"

    # Found the title and it has a rating
    return "matched"


# Run the function on every row's key and store the results
statuses = []
for key in main_keys:
    statuses.append(imdb_status(key))

result["imdb_match_status"] = statuses

Reading IMDb ratings...


In [8]:
print(result["imdb_match_status"].value_counts())
print(result["imdb_rating"].notna().sum(), "rows have a rating")

imdb_match_status
matched              10986
no_match              2931
ambiguous              792
matched_no_rating      174
Name: count, dtype: int64
10986 rows have a rating


**# PART 6: Match Netflix "date added" values for movies and tv shows**

In [9]:
print("Matching Netflix added dates...")

# Step 1: Build a key for every row in the Netflix catalog
netflix_keys = make_keys(
    netflix["type"].str.strip(),   #"Movie" or "TV Show"
    netflix["title"],
    netflix["release_year"],
)

# Step 2: Turn the "date_added" text into real dates
parsed_dates = pd.to_datetime(
    netflix["date_added"].str.strip(),
    format="%B %d, %Y",
    errors="coerce", #if a date is blank or doesn't fit, use NaT
)

# Step 3: Collect candidates for each key
netflix_candidates = defaultdict(set)

for key, show_id, date in zip(netflix_keys, netflix["show_id"], parsed_dates):
    # Ignore Netflix rows whose key isn't in OUR data
    if key not in target_keys:
        continue

    if pd.notna(date):
        date_string = date.strftime("%Y-%m-%d")
    else:
        date_string = None

    # Save the pair, where an identical duplicate row collapses into one entry
    # Two different IDs, or the same title with two different dates, stay as two entries (= ambiguous)
    netflix_candidates[key].add((show_id, date_string))

# Step 4: Work out which of OUR rows are on Netflix
# main["Netflix"] holds 1 if original data says the title is on Netflix, 0 if not
is_netflix = pd.to_numeric(main["Netflix"], errors="coerce").eq(1)

# Step 5: Decide a date and a status for every row
added_dates = []
netflix_statuses = []

# Go through each row's key with its True/False Netflix flag
for key, eligible in zip(main_keys, is_netflix):
    # Look up this key's candidates (an empty set if none were found)
    candidates = netflix_candidates.get(key, set())

    # Case 1: our data says it's not on Netflix, so no date is expected
    if not eligible:
        date_string = None
        status = "not_netflix"

    # Case 2: it should be on Netflix, but the catalog has no such title
    elif len(candidates) == 0:
        date_string = None
        status = "no_match"

    # Case 3: more than one candidate, so we refuse to guess
    elif len(candidates) > 1:
        date_string = None
        status = "ambiguous"

    # Case 4: exactly one candidate
    else:
        _, date_string = next(iter(candidates))

        # Title matched, but its date could be blank
        if date_string is None:
            status = "missing_date"
        else:
            status = "matched"

    added_dates.append(date_string)
    netflix_statuses.append(status)

# Step 6: Add the new columns to "result"
# Convert date texts back into real dates
result["date_added"] = pd.to_datetime(added_dates, errors="coerce")
# Pull out just the year
result["year_added"] = result["date_added"].dt.year.astype("Int64")
# Status column, so we can see why each row has or lacks a date
result["netflix_match_status"] = netflix_statuses

Matching Netflix added dates...


In [10]:
print(result["netflix_match_status"].value_counts())
print(result["date_added"].notna().sum(), "rows have a date")
print(result.loc[result["netflix_match_status"] == "matched", ["Title", "content_type", "date_added", "year_added"]].head(5))

netflix_match_status
not_netflix    9217
matched        4048
no_match       1614
ambiguous         4
Name: count, dtype: int64
4048 rows have a date
                                      Title content_type date_added  \
0                              The Irishman        Movie 2019-11-27   
1                                    Dangal        Movie 2017-06-21   
2  David Attenborough: A Life on Our Planet        Movie 2020-10-04   
4                                      Roma        Movie 2018-12-14   
5         To All the Boys I've Loved Before        Movie 2018-08-17   

   year_added  
0        2019  
1        2017  
2        2020  
4        2018  
5        2018  


**Add missing columns**

In [11]:
runtime_values = []
genre_values = []

for imdb_id in matched_ids:
    runtime, genre = imdb_info.get(imdb_id, ("", ""))
    runtime_values.append(runtime)
    genre_values.append(genre)

result["runtime_minutes"] = runtime_values
result["genres"] = genre_values

netflix_country = {}
for key, country in zip(netflix_keys, netflix["country"]):
    if key in target_keys:
        netflix_country[key] = country.strip()

country_values = []
for key, status in zip(main_keys, result["netflix_match_status"]):
    if status == "matched":
        country_values.append(netflix_country.get(key, ""))
    else:
        country_values.append("")

result["country"] = country_values

print(list(result.columns))
print((result["runtime_minutes"] != "").sum(), "rows have a runtime")
print((result["genres"] != "").sum(), "rows have genres")
print((result["country"] != "").sum(), "rows have a country")

['Unnamed: 0', 'ID', 'Title', 'Year', 'Age', 'Rotten Tomatoes', 'Netflix', 'Hulu', 'Prime Video', 'Disney+', 'Type', 'content_type', 'record_id', 'imdb_id', 'imdb_rating', 'imdb_num_votes', 'imdb_match_status', 'date_added', 'year_added', 'netflix_match_status', 'runtime_minutes', 'genres', 'country']
10375 rows have a runtime
11117 rows have genres
3753 rows have a country


**Safety Checks**

In [12]:
# Check that the final table has exactly as many rows as the two original files combined
# (9,515 movies + 5,368 TV = 14,883)
assert len(result) == len(movies) + len(tv), "Row count changed."

# Check that every IMDb rating is between 1 and 10
assert result["imdb_rating"].dropna().between(1, 10).all(), "A rating is outside 1-10."

# Only reach this line if both checks passed
print("Checks passed.")

Checks passed.


**Part 7: Export, summarize, and download**

In [13]:
# Save table to a CSV file
output_path = folder / "streaming_movies_tv_merged.csv"

result.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
    na_rep="",
)

# Summary
print("MERGE COMPLETE")
print(f"Movie rows:    {len(movies):,}")
print(f"TV show rows:  {len(tv):,}")
print(f"Combined rows: {len(result):,}")
print(f"IMDb ratings:  {result['imdb_rating'].notna().sum():,}")
print(f"Netflix dates: {result['date_added'].notna().sum():,}")
# How many rows landed in each IMDb status, split by Movie & TV Show
print("\nIMDb match counts:")
print(result.groupby(["content_type", "imdb_match_status"]).size().to_string())
print("\nNetflix match counts:")
print(result.groupby(["content_type", "netflix_match_status"]).size().to_string())

display(result.groupby("content_type", sort=False).head(3))

print(f"\nSaved: {output_path.name}")
#files.download(str(output_path))

MERGE COMPLETE
Movie rows:    9,515
TV show rows:  5,368
Combined rows: 14,883
IMDb ratings:  10,986
Netflix dates: 4,048

IMDb match counts:
content_type  imdb_match_status
Movie         ambiguous             691
              matched              7059
              matched_no_rating      61
              no_match             1704
TV Show       ambiguous             101
              matched              3927
              matched_no_rating     113
              no_match             1227

Netflix match counts:
content_type  netflix_match_status
Movie         ambiguous                  3
              matched                 2993
              no_match                 699
              not_netflix             5820
TV Show       ambiguous                  1
              matched                 1055
              no_match                 915
              not_netflix             3397


,Unnamed: 0,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,...,imdb_id,imdb_rating,imdb_num_votes,imdb_match_status,date_added,year_added,netflix_match_status,runtime_minutes,genres,country
0,0,1,The Irishman,2019,18+,98/100,1,0,0,0,...,tt1302006,7.8,482099,matched,2019-11-27,2019,matched,209,"Biography,Crime,Drama",United States
1,1,2,Dangal,2016,7+,97/100,1,0,0,0,...,tt5074352,8.3,249711,matched,2017-06-21,2017,matched,161,"Action,Biography,Drama",India
2,2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,...,tt11989890,8.9,36517,matched,2020-10-04,2020,matched,83,"Biography,Documentary","United States, United Kingdom"
9515,0,1,Breaking Bad,2008,18+,100/100,1,0,0,0,...,tt0903747,9.5,2699258,matched,NaT,<NA>,no_match,48,"Crime,Drama,Thriller",
9516,1,2,Stranger Things,2016,16+,96/100,1,0,0,0,...,tt4574334,8.6,1751352,matched,NaT,<NA>,no_match,60,"Drama,Fantasy,Horror",
9517,2,3,Attack on Titan,2013,18+,95/100,1,1,0,0,...,tt2560140,9.1,756966,matched,2021-06-02,2021,matched,24,"Action,Adventure,Animation",Japan



Saved: streaming_movies_tv_merged.csv


In [15]:
#check missing data
print(result.duplicated(subset=["Title", "Year", "content_type"]).sum())

0
